# Colab 20 — Recalibration readout: fix the high-band compression WITHOUT retraining

colab19 showed the within-high-band value is **present but unread**: the 3-bin CE head
saturates the L2-sim readout (~0.8) while the *geometry* stays monotone (synthetic AA
isotonic OOF RMSE 0.038, Spearman 0.847). So the fix is a post-hoc **monotone calibrator**
on the **frozen** encoder — no retraining. This notebook fits it and checks it transfers.

**The claim being tested: one encoder + one readout, both alphabet-agnostic.**
- **Encoder:** the AA encoder, frozen, applied zero-shot to AA / SS / 3Di feeds.
- **Deployed readout = UNIVERSAL calibrator:** isotonic, fit on **full-range synthetic AA**
  (`cath_synth_fullrange_aa.csv.gz`) + **identity anchors at (1.0, 1.0)** (so `f(1.0)=1.0`),
  `y_max=1.0`. A PCHIP monotone spline is saved as the portable closed form.
- **Tested ONLY on natural** AA/SS/3Di pairs (never fit on them) -> proves *transfer*, not
  memorisation.

**Numbers reported (per feed):** raw vs calibrated RMSE; **full-range as support, high-band
(`>= 0.70`) RMSE as the headline** (that's where the retrieval/value claim lives). Full-range
*natural AA* is powered (~5000 eval pairs), so we get a powered AA calibration number even
though high-band AA stays n=5.

**Diagnostics (NOT deployed):** a high-band-only synthetic calibrator (upper bound if you only
cared about `>= 0.70`) and per-feed fit-on-natural calibrators (upper bound if tuned per
representation) — both compared to the universal on high-band RMSE.

Deferred: colab21 full-alphabet / natural-language probe; colab22 masked-adaptive-pooling.


## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy --quiet

In [ ]:
import torch, json
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold
from scipy.stats import spearmanr
from scipy.interpolate import PchipInterpolator
from rapidfuzz.distance import Levenshtein as RFLevenshtein

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
print('device:', device)

## 2. Constants (encoder recipe = colab17b/colab19; AA encoder only)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN, MAX_LEN_SEQ, MAX_LEN = 50, 200, 200
N_TRAIN, NUM_EPOCHS, BATCH_SIZE, K = 30000, 30, 128, 16
BAND_LOW_AA, BAND_HIGH = 0.30, 0.70
BIN_NAMES = ['far', 'mid', 'high']

N_FOLDS = 5
N_ANCHOR = 200            # identity anchor weight at (pred=1.0, true=1.0)
SEED_TRAIN_AA = 42
SEED_FULLRANGE = 999      # must match the committed full-range set
SEED_SYNTH = 12345        # must match the committed high-sim set

FULLRANGE_PATH = os.path.join(DATA_DIR, 'cath_synth_fullrange_aa.csv.gz')
HIGHSIM_PATH   = os.path.join(DATA_DIR, 'cath_synth_highsim_aa.csv.gz')
N_FULLRANGE, N_HIGHSIM = 8000, 5000
LOAD_FROZEN = True

## 3. Helpers (RNG threaded) + architecture + AA training

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(s): return all(c in AA_SET for c in s)
def is_standard_ss(s): return all(c in SS_SET for c in s)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def make_full_range_pairs(alphabet, n_pairs, rng):
    pairs = []; attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0.0, 1.0)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

def bin_idx_for(x, band_low):
    return 0 if x < band_low else (1 if x < BAND_HIGH else 2)


class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K, self.pad_idx = K, pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, 3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(K)
        self.fc = nn.Linear(hidden2 * K, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1); h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)

class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
                                  nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b): return self.head(torch.abs(self.encoder(a) - self.encoder(b)))

class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]

def train_aa_encoder(train_seed=SEED_TRAIN_AA, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed); rng = np.random.default_rng(train_seed)
    print(f'===== Training AA encoder (seed={train_seed}) =====')
    pairs = make_full_range_pairs(AA_ALPHABET, N_TRAIN, rng)
    dl = DataLoader(PairDatasetCE(pairs, BAND_LOW_AA), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y); opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval(); return model

model_aa = train_aa_encoder()

## 4. Data load — pool + eval (natural test sets)

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
prot_df = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')
prot_df = prot_df[prot_df['aa_seq'].apply(is_standard_aa) & prot_df['ss_seq'].apply(is_standard_ss)]
prot_df = prot_df[prot_df['aa_seq'].str.len() == prot_df['ss_seq'].str.len()]
prot_df['aa_len'] = prot_df['aa_seq'].str.len()
RESCUED = {'4z0mC02', '3qkaE02'}
in_range = prot_df['aa_len'].between(MIN_LEN, MAX_LEN)
prot_df = prot_df[in_range | prot_df['domain_id'].isin(RESCUED)].reset_index(drop=True)
id_to_aa = dict(zip(prot_df['domain_id'], prot_df['aa_seq']))
id_to_ss = dict(zip(prot_df['domain_id'], prot_df['ss_seq']))
all_domains = list(prot_df['domain_id'])
print('pool:', len(prot_df))

eval_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))
id_to_3di = dict(zip(seqs3['domain_id'], seqs3['3di'].astype(str)))
def pair_3di(a, b):
    return norm_lev(id_to_3di[a], id_to_3di[b]) if (a in id_to_3di and b in id_to_3di) else np.nan
eval_df['3di_score'] = [pair_3di(a, b) for a, b in zip(eval_df['domain_a'], eval_df['domain_b'])]
SCORE_COL = {'AA': 'aa_score', 'SS': 'ss_score', '3Di': '3di_score'}
LOOKUP = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}

## 5. Calibrator machinery

`fit_calibrator`: isotonic with `y_max=1.0` + `N_ANCHOR` identity anchors at (1.0, 1.0) so
`f(1.0)=1.0`. `oof_rmse`: CV with anchors added to each *train* fold only (never the test
fold). `pchip_from`: portable monotone closed form sampled off the isotonic curve.


In [ ]:
@torch.no_grad()
def pred_sim(model, A, B, batch=512):
    model.eval(); out = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        out.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def _with_anchors(pred, true, n=N_ANCHOR):
    return (np.concatenate([pred, np.ones(n)]), np.concatenate([true, np.ones(n)]))

def fit_calibrator(pred, true, anchor=True):
    p, t = (_with_anchors(np.asarray(pred,float), np.asarray(true,float)) if anchor
            else (np.asarray(pred,float), np.asarray(true,float)))
    return IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0).fit(p, t)

def rmse(true, pred):
    return float(np.sqrt(np.mean((np.asarray(true,float) - np.asarray(pred,float)) ** 2)))

def oof_rmse(pred, true, n_folds=N_FOLDS, seed=SEED, anchor=True):
    pred = np.asarray(pred, float); true = np.asarray(true, float); n = len(true)
    if n < n_folds * 2: return np.nan
    oof = np.full(n, np.nan)
    for tr, te in KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(pred):
        p, t = (_with_anchors(pred[tr], true[tr]) if anchor else (pred[tr], true[tr]))
        ir = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0).fit(p, t)
        oof[te] = ir.predict(pred[te])
    return rmse(true, oof)

def apply_cal(cal, pred):
    return np.clip(cal.predict(np.asarray(pred, float)), 0.0, 1.0)

def pchip_from(cal, n_knots=40):
    xs = np.linspace(0.0, 1.0, n_knots); ys = apply_cal(cal, xs)
    ys = np.maximum.accumulate(ys)              # enforce monotone for PChip safety
    return PchipInterpolator(xs, ys), xs, ys

## 6. Fit the UNIVERSAL calibrator on full-range synthetic AA (frozen)

In [ ]:
if LOAD_FROZEN and os.path.exists(FULLRANGE_PATH):
    fr = pd.read_csv(FULLRANGE_PATH)
    print(f'Loaded FROZEN full-range set: {FULLRANGE_PATH} (n={len(fr)})')
else:
    rng = np.random.default_rng(SEED_FULLRANGE)
    pr = make_full_range_pairs(AA_ALPHABET, N_FULLRANGE, rng)
    fr = pd.DataFrame(pr, columns=['seq_a', 'seq_b', 'norm_lev']); fr.to_csv(FULLRANGE_PATH, index=False)
    print(f'Generated + wrote full-range set -> {FULLRANGE_PATH} (commit to freeze)')

fr_true = fr['norm_lev'].values
fr_pred = pred_sim(model_aa, list(fr['seq_a']), list(fr['seq_b']))
print(f'  full-range: raw L2-sim mean={fr_pred.mean():.3f}, normLev in [{fr_true.min():.3f}, {fr_true.max():.3f}]')

UNIVERSAL = fit_calibrator(fr_pred, fr_true, anchor=True)   # <-- DEPLOYED readout
pchip, knots_x, knots_y = pchip_from(UNIVERSAL)
fr_oof = oof_rmse(fr_pred, fr_true)
print(f'  UNIVERSAL calibrator fit (isotonic + identity anchor). full-range OOF RMSE = {fr_oof:.4f}')
print(f'  identity check: f(1.0) = {float(apply_cal(UNIVERSAL,[1.0])[0]):.4f}  (target 1.000)')

## 7. Apply UNIVERSAL calibrator — synthetic high-sim (held-out) + natural feeds

In [ ]:
# (a) held-out synthetic high-sim (frozen set, disjoint from full-range fit by seed)
hs = pd.read_csv(HIGHSIM_PATH)
hs_true = hs['norm_lev'].values
hs_pred = pred_sim(model_aa, list(hs['seq_a']), list(hs['seq_b']))
hs_raw, hs_cal = rmse(hs_true, hs_pred), rmse(hs_true, apply_cal(UNIVERSAL, hs_pred))
print(f'synthetic high-sim (held-out, n={len(hs)}): raw RMSE={hs_raw:.4f} -> calibrated={hs_cal:.4f}')

# (b) natural feeds — AA-encoder zero-shot, universal calibrator, NEVER fit on these
def natural_cell(feed):
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    inpool = set(all_domains) & set(lk)
    sub = eval_df.dropna(subset=[sc])
    sub = sub[sub['domain_a'].isin(inpool) & sub['domain_b'].isin(inpool)]
    full = sub.copy(); hi = sub[sub[sc] >= BAND_HIGH]
    out = {'feed': feed, 'n_full': len(full), 'n_high': len(hi)}
    if len(full):
        t = full[sc].values; p = pred_sim(model_aa, [lk[d] for d in full['domain_a']],
                                                    [lk[d] for d in full['domain_b']])
        out['raw_full'] = rmse(t, p); out['cal_full'] = rmse(t, apply_cal(UNIVERSAL, p))
    if len(hi):
        t = hi[sc].values; p = pred_sim(model_aa, [lk[d] for d in hi['domain_a']],
                                                  [lk[d] for d in hi['domain_b']])
        out['raw_high'] = rmse(t, p); out['cal_high'] = rmse(t, apply_cal(UNIVERSAL, p))
        out['hi_pred'] = p; out['hi_true'] = t
    return out

natural = {f: natural_cell(f) for f in ['AA', 'SS', '3Di']}
print(f'\n{"feed":<6}{"n_full":>7}{"raw_full":>10}{"cal_full":>10}{"n_high":>8}{"raw_high":>10}{"cal_high":>10}')
for f in ['AA', 'SS', '3Di']:
    d = natural[f]
    print(f"{f:<6}{d['n_full']:>7}{d.get('raw_full',np.nan):>10.4f}{d.get('cal_full',np.nan):>10.4f}"
          f"{d['n_high']:>8}{d.get('raw_high',float('nan')):>10.4f}{d.get('cal_high',float('nan')):>10.4f}")

## 8. HEADLINE table — universal calibrator (high-band RMSE = the thesis number)

Full-range = support (deployable, powered even for AA). High-band `>= 0.70` = the headline,
because that is where the retrieval/value claim lives. AA high-band stays n=5 (caveat);
AA full-range is powered.


In [ ]:
print('=' * 84)
print('UNIVERSAL calibrator (one AA-synthetic fit, applied unchanged to every feed)')
print('=' * 84)
print(f'{"feed":<8}{"high-band n":>12}{"raw RMSE":>11}{"CALIBRATED":>12}{"  (full-range cal, n)":>24}')
print('-' * 84)
print(f'{"AA-synth":<8}{len(hs):>12}{hs_raw:>11.4f}{hs_cal:>12.4f}{"  (fit substrate=full-range)":>24}')
for f in ['AA', 'SS', '3Di']:
    d = natural[f]
    fr_note = f"  ({d.get('cal_full',float('nan')):.4f}, n={d['n_full']})"
    print(f"{f+'-nat':<8}{d['n_high']:>12}{d.get('raw_high',float('nan')):>11.4f}"
          f"{d.get('cal_high',float('nan')):>12.4f}{fr_note:>24}")
print('\nraw = uncalibrated L2-sim vs normLev (the ~0.8 saturation). CALIBRATED = after the')
print('frozen-encoder monotone readout. The drop raw->calibrated IS the fix, no retraining.')

## 9. DIAGNOSTIC calibrators (NOT deployed): high-band-only + per-feed upper bounds

In [ ]:
# (i) high-band-only synthetic calibrator: fit on frozen high-sim AA, OOF — upper bound if
#     we only cared about >= 0.70.
hb_oof = oof_rmse(hs_pred, hs_true)
# (ii) per-feed fit-on-natural calibrators (CV): upper bound if the readout were tuned per rep.
print(f'{"feed":<8}{"universal_high":>16}{"perfeed_high(CV)":>18}{"gain":>8}')
print('-' * 50)
print(f'{"AA-synth":<8}{hs_cal:>16.4f}{hb_oof:>18.4f}{hs_cal-hb_oof:>8.4f}   <- high-band-only synth bound')
for f in ['AA', 'SS', '3Di']:
    d = natural[f]
    if d['n_high'] >= N_FOLDS * 2:
        pf = oof_rmse(d['hi_pred'], d['hi_true'])
        print(f"{f+'-nat':<8}{d.get('cal_high',float('nan')):>16.4f}{pf:>18.4f}"
              f"{d.get('cal_high',float('nan'))-pf:>8.4f}")
    else:
        print(f"{f+'-nat':<8}{d.get('cal_high',float('nan')):>16.4f}{'n/a (n<10)':>18}{'':>8}")
print('\nuniversal_high = deployed universal calibrator on this feed (high-band).')
print('perfeed_high(CV) = isotonic fit ON this feed (natural, CV) = per-rep upper bound.')
print('Small gain => the universal alphabet-agnostic readout loses little vs per-rep tuning.')

## 10. Identity sanity check — (X, X) must calibrate to ~1.0

In [ ]:
rng_id = np.random.default_rng(0)
sample_ids = list(rng_id.choice(all_domains, size=min(50, len(all_domains)), replace=False))
for feed, lk in [('AA', id_to_aa), ('SS', id_to_ss), ('3Di', id_to_3di)]:
    seqs = [lk[d] for d in sample_ids if d in lk]
    p = pred_sim(model_aa, seqs, seqs)               # (X, X)
    c = apply_cal(UNIVERSAL, p)
    print(f'{feed:<5} identity pairs n={len(seqs)}: raw L2-sim mean={p.mean():.4f} (min {p.min():.4f}), '
          f'calibrated mean={c.mean():.4f} (min {c.min():.4f})')
assert apply_cal(UNIVERSAL, [1.0])[0] >= 0.999, 'identity anchor failed: f(1.0) < 0.999'
print('OK: identity anchor holds (f(1.0) ~ 1.0).')

## 11. Before/after calibration plot + save artifacts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
panels = [('AA-synth high-sim', hs_pred, hs_true)]
panels += [(f'{f}-nat high-sim', natural[f].get('hi_pred'), natural[f].get('hi_true'))
           for f in ['SS', '3Di']]
gx = np.linspace(0.5, 1.0, 100)
for ax, (title, p, t) in zip(axes, panels):
    if p is None or len(p) == 0:
        ax.set_title(title + ' (no data)'); continue
    ax.scatter(p, t, s=6, alpha=0.2, label='raw L2-sim')
    ax.scatter(apply_cal(UNIVERSAL, p), t, s=6, alpha=0.2, color='g', label='calibrated')
    ax.plot(gx, apply_cal(UNIVERSAL, gx), 'k-', lw=1.5, label='universal calibrator')
    ax.plot([0.5, 1.0], [0.5, 1.0], 'r--', lw=1)
    ax.set_xlim(0.5, 1.0); ax.set_ylim(0.65, 1.01)
    ax.set_xlabel('predicted'); ax.set_ylabel('true normLev'); ax.set_title(title); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('colab20_calibration.png', dpi=120); plt.show()

# Save the deployed calibrator (isotonic knots + PCHIP) so the readout is reproducible/citable.
# Export as a dense grid sample (version-independent; reconstruct via np.interp or PCHIP).
grid_x = np.linspace(0.0, 1.0, 201)
grid_y = apply_cal(UNIVERSAL, grid_x).tolist()
cal_params = {'type': 'isotonic+identity_anchor', 'y_max': 1.0, 'n_anchor': N_ANCHOR,
              'fit_set': 'cath_synth_fullrange_aa.csv.gz', 'encoder_seed': SEED_TRAIN_AA,
              'grid_x': grid_x.tolist(), 'grid_y': grid_y,
              'pchip_knots_x': knots_x.tolist(), 'pchip_knots_y': knots_y.tolist()}
with open('colab20_universal_calibrator.json', 'w') as fh:
    json.dump(cal_params, fh)
print('saved colab20_calibration.png, colab20_universal_calibrator.json')

## How to read this notebook

**The fix works if `CALIBRATED` << `raw` and the universal high-band RMSE is ~0.04.** That
closes the colab19 arc: compression diagnosed, then removed with a frozen-encoder monotone
readout — no retraining.

**Alphabet-agnostic readout (the claim):** one calibrator fit on synthetic AA, applied
unchanged to SS/3Di natural pairs. If `universal_high` is close to the per-feed `perfeed_high`
upper bound, the readout transfers across representations like the encoder does. Expect SS->3Di
to be the weakest cell (colab19: 0.051-0.054) — report 3Di as moderate evidence (n=38).

**Honest scope:** high-band AA is n=5 (full-range AA is powered, ~5000). The high-band-only
synthetic calibrator is an upper bound, NOT the deployed readout (it assumes you already know
the pair is >= 0.70). Deployed = universal full-range. Per-feed fit-on-natural is a diagnostic
ceiling, not a headline (fitting and reporting on the same representation).
